# Initialization

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

from skopt import gp_minimize
from skopt.space import Real, Categorical
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, RBF

import warnings
import numpy as np
import pandas as pd
from pathlib import Path

## Directories

In [2]:
data_dir = Path("data")
Path.mkdir(data_dir, exist_ok=True)

plot_dir = Path("plots")
Path.mkdir(plot_dir, exist_ok=True)

log_dir = Path("logs")
Path.mkdir(log_dir, exist_ok=True)

## Data reading

In [3]:
data_file = "Dataset_SL.xlsx"

### Define experimental data

In [6]:
# 1. Define the experimental data
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")

In [7]:
# make the other columns as floats
experiment_data = experiment_data.astype(
    {
        "salt_concentration": float,
        "water_to_cement_ratio": float,
        "antisettling_concentration": float,
        "E_d": float,
        "KPI": float,
    }
)

# get optimization round for later use in saving
opt_round = experiment_data["opt_round"].iloc[-1]

In [8]:
experiment_data

,opt_round,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,E_d,KPI
0,0,MgSO4,0.5,1.0,0.0,NaN,NaN,NaN,27.764640,9.597843
1,0,CaCl2,0.5,1.0,0.0,NaN,NaN,NaN,87.715788,2.519732
2,0,SrBr2,0.5,1.0,0.0,NaN,NaN,NaN,95.547176,53.682419
3,0,MgCl2,0.5,1.0,0.0,NaN,NaN,NaN,82.071086,2.933749
4,0,CuSO4,0.5,1.0,0.0,NaN,NaN,NaN,5.954210,100.212303
5,0,Al2(SO4)3,0.5,1.0,0.0,NaN,NaN,NaN,14.944089,15.671044
6,0,K2CO3,0.5,1.0,0.0,NaN,NaN,NaN,23.614710,38.117562
7,0,KAl(SO4)2,0.5,1.0,0.0,NaN,NaN,NaN,26.974859,7.504554
8,0,LiCl,0.5,1.0,0.0,NaN,NaN,NaN,158.387724,53.476998
9,0,Mg(NO3)2,0.5,1.0,0.0,NaN,NaN,NaN,55.930893,9.326240


# Search space

### Category encoding

In [9]:
cat_order = [
    "Al2(SO4)3",
    "CaCl2",
    "CuSO4",
    "K2CO3",
    "KAl(SO4)2",
    "LiCl",
    "Mg(NO3)2",
    "MgCl2",
    "MgSO4",
    "SrBr2",
    "Zn(NO3)2",
]

cat_mapping = {name: i for i, name in enumerate(cat_order)}

df_enc = experiment_data.copy()
df_enc["category_encoded"] = experiment_data["salt"].map(cat_mapping)
df_enc

,opt_round,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,E_d,KPI,category_encoded
0,0,MgSO4,0.5,1.0,0.0,NaN,NaN,NaN,27.764640,9.597843,8
1,0,CaCl2,0.5,1.0,0.0,NaN,NaN,NaN,87.715788,2.519732,1
2,0,SrBr2,0.5,1.0,0.0,NaN,NaN,NaN,95.547176,53.682419,9
3,0,MgCl2,0.5,1.0,0.0,NaN,NaN,NaN,82.071086,2.933749,7
4,0,CuSO4,0.5,1.0,0.0,NaN,NaN,NaN,5.954210,100.212303,2
5,0,Al2(SO4)3,0.5,1.0,0.0,NaN,NaN,NaN,14.944089,15.671044,0
6,0,K2CO3,0.5,1.0,0.0,NaN,NaN,NaN,23.614710,38.117562,3
7,0,KAl(SO4)2,0.5,1.0,0.0,NaN,NaN,NaN,26.974859,7.504554,4
8,0,LiCl,0.5,1.0,0.0,NaN,NaN,NaN,158.387724,53.476998,5
9,0,Mg(NO3)2,0.5,1.0,0.0,NaN,NaN,NaN,55.930893,9.326240,6


In [10]:
search_space = [
    Categorical(cat_order, name="salt"),  # salt names
    Real(0.1, 0.9, name="salt_concentration"),
    Real(0.7, 1.5, name="water_to_cement_ratio"),
    Real(0.0, 3.0, name="antisettling_concentration"),
]

# Optimization

In [11]:
salt_dummies = pd.get_dummies(df_enc["salt"], prefix="salt").reindex(
    columns=[f"salt_{cat}" for cat in cat_order], fill_value=0
)

In [12]:
X = pd.concat(
    [
        salt_dummies,
        df_enc[
            [
                "salt_concentration",
                "water_to_cement_ratio",
                "antisettling_concentration",
            ]
        ],
    ],
    axis=1,
).to_numpy()
y_energy = df_enc["E_d"].values
y_kpi = df_enc["KPI"].values

In [13]:
salt_dummies

,salt_Al2(SO4)3,salt_CaCl2,salt_CuSO4,salt_K2CO3,salt_KAl(SO4)2,salt_LiCl,salt_Mg(NO3)2,salt_MgCl2,salt_MgSO4,salt_SrBr2,salt_Zn(NO3)2
0,False,False,False,False,False,False,False,False,True,False,False
1,False,True,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,True,False,False,False
4,False,False,True,False,False,False,False,False,False,False,False
5,True,False,False,False,False,False,False,False,False,False,False
6,False,False,False,True,False,False,False,False,False,False,False
7,False,False,False,False,True,False,False,False,False,False,False
8,False,False,False,False,False,True,False,False,False,False,False
9,False,False,False,False,False,False,True,False,False,False,False


## Gaussian Process

In [14]:
def fit_gp_models(X, y, kernel):
    """Fits Gaussian Process models to the given experimental data."""  # noqa

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        alpha=1e-3,
    )

    gp.fit(X, y)

    return gp

### Matern kernel

In [15]:
gp_energy_matern = fit_gp_models(
    X,
    -y_energy,
    Matern(length_scale=5e-3, length_scale_bounds=(1e-8, 10.0), nu=2.5),  # noqa
)

In [16]:
gp_kpi_matern = fit_gp_models(
    X,
    y_kpi,
    Matern(length_scale=5e-3, length_scale_bounds=(1e-8, 10.0), nu=2.5),  # noqa
)

### RBF kernel

In [17]:
gp_energy_rbf = fit_gp_models(
    X,
    -y_energy,
    RBF(
        length_scale=5e-3,
        length_scale_bounds=(1e-8, 10.0),
    ),
)

In [18]:
gp_kpi_rbf = fit_gp_models(
    X,
    y_kpi,
    RBF(
        length_scale=5e-3,
        length_scale_bounds=(1e-8, 10.0),
    ),
)

# Bayesian Optimization

In [19]:
def encode_input(x):
    # x[0] is salt name (e.g., "MgSO4")
    salt_vector = np.zeros(len(cat_mapping))
    salt_index = cat_mapping[x[0]]
    salt_vector[salt_index] = 1

    # concatenate with continuous features
    return np.concatenate([salt_vector, np.array(x[1:])])

In [29]:
def suggest_new_samples(
    gp_model: GaussianProcessRegressor,
    obj_func: str,
    print_points: bool = False,
    verbose: bool = False,
    mute_warnings: bool = True,
):
    """Suggests new samples based on the trained GP model using different acquisition functions."""  # noqa

    acq_functions = ["EI", "PI", "LCB_low", "LCB_med", "LCB_high"]
    kappa_values = {
        "LCB_low": 1.0,
        "LCB_med": 2.5,
        "LCB_high": 5.0,
    }
    new_samples = []

    with warnings.catch_warnings():
        if mute_warnings:
            print("Warnings are muted!")
            warnings.simplefilter("ignore")

        for i, acquisition in enumerate(acq_functions):
            res = gp_minimize(
                lambda x: gp_model.predict([encode_input(x)])[
                    0
                ],  # Optimize our surrogate model # noqa
                dimensions=search_space,  # pass our search space
                base_estimator=gp_model,
                acq_func=(
                    "LCB"
                    if acquisition in ["LCB_low", "LCB_med", "LCB_high"]
                    else acquisition
                ),  # define the acquisition function
                kappa=kappa_values.get(
                    acquisition
                ),  # define the custom k value if acquisition if "LCB" # noqa
                xi=0.05,
                n_calls=40,
                verbose=verbose,
                n_jobs=6,
            )

            # Convert numerical salt encoding back to categorical
            suggested = res.x
            if print_points:
                print(suggested)

            suggested.append(str(gp_model.kernel).split("(")[0])
            suggested.append(str(acquisition))
            suggested.append(str(obj_func))

            # save suggested samples in a list
            new_samples.append(suggested)

    # create a new dataframe with suggested samples
    columns = [
        "salt",
        "salt_concentration",
        "water_to_cement_ratio",
        "antisettling_concentration",
        "kernel",
        "acquisition",
        "obj_func",
    ]

    df = pd.DataFrame(new_samples, columns=columns)

    # round float values to 3rd decimal place
    df[df.select_dtypes(include="float").columns] = df.select_dtypes(
        include="float"
    ).round(3)

    return df

### Generate 10 new samples for Energy Density optimization

In [30]:
# Matérn kernel
new_samples_energy_matern = suggest_new_samples(
    gp_energy_matern, obj_func="E_d", verbose=True, mute_warnings=True
)

Warnings are muted!
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0010
Function value obtained: -71.6662
Current minimum: -71.6662
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -18.1907
Current minimum: -71.6662
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -46.2839
Current minimum: -71.6662
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0010
Function value obtained: -50.1032
Current minimum: -71.6662
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -27.1229
Current minimum: -71.

In [31]:
new_samples_energy_matern

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func
0,SrBr2,0.517,0.910,0.00,Matern,EI,E_d
1,MgCl2,0.315,0.997,0.44,Matern,PI,E_d
2,LiCl,0.432,1.500,0.00,Matern,LCB_low,E_d
3,LiCl,0.900,0.700,0.00,Matern,LCB_med,E_d
4,CaCl2,0.470,0.941,0.00,Matern,LCB_high,E_d


In [32]:
# RBF kernel
new_samples_energy_rbf = suggest_new_samples(
    gp_energy_rbf, obj_func="E_d", verbose=True, mute_warnings=True
)

Warnings are muted!
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -27.9048
Current minimum: -27.9048
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -17.3134
Current minimum: -27.9048
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -28.2485
Current minimum: -28.2485
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -10.2883
Current minimum: -28.2485
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -77.7035
Current minimum: -77.

In [33]:
new_samples_energy_rbf

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func
0,MgCl2,0.279,1.044,0.0,RBF,EI,E_d
1,SrBr2,0.450,0.920,0.0,RBF,PI,E_d
2,CaCl2,0.846,0.980,3.0,RBF,LCB_low,E_d
3,SrBr2,0.900,1.088,3.0,RBF,LCB_med,E_d
4,CaCl2,0.739,0.854,0.0,RBF,LCB_high,E_d


### Generate 10 new samples for economic KPI optimization

In [34]:
# Matérn kernel
new_samples_kpi_matern = suggest_new_samples(
    gp_kpi_matern, obj_func="KPI", verbose=True, mute_warnings=True
)

Warnings are muted!
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 12.6381
Current minimum: 12.6381
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 58.5774
Current minimum: 12.6381
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 52.0904
Current minimum: 12.6381
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 20.2485
Current minimum: 12.6381
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 49.0816
Current minimum: 12.6381
Itera

In [35]:
new_samples_kpi_matern

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func
0,MgCl2,0.312,1.027,0.458,Matern,EI,KPI
1,CaCl2,0.416,1.387,0.000,Matern,PI,KPI
2,MgCl2,0.366,1.500,0.378,Matern,LCB_low,KPI
3,MgCl2,0.100,1.500,0.678,Matern,LCB_med,KPI
4,MgCl2,0.511,1.425,0.433,Matern,LCB_high,KPI


In [36]:
# RBF kernel
new_samples_kpi_rbf = suggest_new_samples(
    gp_kpi_rbf, obj_func="KPI", verbose=True, mute_warnings=True
)  # noqa

Warnings are muted!
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 32.0777
Current minimum: 32.0777
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 32.0777
Current minimum: 32.0777
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 32.0777
Current minimum: 32.0777
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 32.0777
Current minimum: 32.0777
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: 32.0776
Current minimum: 32.0776
Itera

In [37]:
new_samples_kpi_rbf

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func
0,Mg(NO3)2,0.237,1.286,1.719,RBF,EI,KPI
1,Mg(NO3)2,0.485,1.077,0.027,RBF,PI,KPI
2,KAl(SO4)2,0.854,1.394,0.590,RBF,LCB_low,KPI
3,MgSO4,0.557,0.937,0.085,RBF,LCB_med,KPI
4,MgSO4,0.706,1.452,0.254,RBF,LCB_high,KPI


## Saving the new suggestions in the original excel sheet

In [38]:
# Combine sets of new samples
new_samples = pd.concat(
    [
        new_samples_energy_matern,
        new_samples_energy_rbf,
        new_samples_kpi_matern,
        new_samples_kpi_rbf,
    ],
    ignore_index=True,
)

new_samples.insert(0, "opt_round", opt_round + 1)  # add optimization round

# Read the existing Excel sheet into a DataFrame
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")
# Append the new data to the existing DataFrame
combined_data = pd.concat(
    [experiment_data, new_samples], ignore_index=True, axis=0
)  # noqa

# Write the updated DataFrame back to the same Excel sheet
with pd.ExcelWriter(
    data_dir / data_file, engine="openpyxl", mode="a", if_sheet_exists="replace"  # noqa
) as writer:
    combined_data.to_excel(writer, sheet_name="Datasheet", index=False)

print(
    "New batch of 20 samples suggested. Please conduct experiments and update the dataset."  # noqa
)

New batch of 20 samples suggested. Please conduct experiments and update the dataset.
